# Bronze ingestion — optimized

Optimizations:
- explicit raw schemas instead of `inferSchema`
- raw values kept as strings; semantic typing stays in Silver
- no pre-write `count()`
- no production preview queries
- diagnostics use one aggregation per table


In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType,
    LongType, DoubleType
)
import json

VOLUME_BASE_PATH = "/Volumes/Data_Lakehouse_Databricks/Bronze/Bronze_Vol/"
CATALOG = "Data_Lakehouse_Databricks"
BRONZE_SCHEMA = "Bronze"
FOLDERS = ["source_erp", "source_crm"]

def raw_string_schema(*columns):
    return StructType([
        StructField(column, StringType(), True)
        for column in columns
    ])

SOURCE_SCHEMAS = {
    "cust_info": raw_string_schema(
        "cst_id", "cst_key", "cst_firstname", "cst_lastname",
        "cst_marital_status", "cst_gndr", "cst_create_date"
    ),
    "prd_info": raw_string_schema(
        "prd_id", "prd_key", "prd_nm", "prd_cost",
        "prd_line", "prd_start_dt", "prd_end_dt"
    ),
    "sales_details": raw_string_schema(
        "sls_ord_num", "sls_prd_key", "sls_cust_id",
        "sls_order_dt", "sls_ship_dt", "sls_due_dt",
        "sls_sales", "sls_quantity", "sls_price"
    ),
    "cust_az12": raw_string_schema("CID", "BDATE", "GEN"),
    "loc_a101": raw_string_schema("CID", "CNTRY"),
    "px_cat_g1v2": raw_string_schema(
        "ID", "CAT", "SUBCAT", "MAINTENANCE"
    ),
}

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{BRONZE_SCHEMA}"
)


In [ ]:
failures = []

for folder in FOLDERS:
    folder_path = f"{VOLUME_BASE_PATH}{folder}/"
    folder_prefix = folder.replace("source_", "")
    print(f"Processing folder: {folder}")

    try:
        files = dbutils.fs.ls(folder_path)
    except Exception as exc:
        failures.append(
            f"{folder}: unable to list source files: {exc}"
        )
        continue

    csv_files = [
        f for f in files
        if f.name.lower().endswith(".csv")
    ]

    for file_info in csv_files:
        source_name = file_info.name[:-4].lower()

        if source_name not in SOURCE_SCHEMAS:
            failures.append(
                f"{file_info.path}: no schema configured "
                f"for {source_name}"
            )
            continue

        table_name = f"bronze_{folder_prefix}_{source_name}"
        full_table_name = (
            f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
        )

        print(f"Reading: {file_info.path}")
        print(f"Target:  {full_table_name}")

        try:
            df = (
                spark.read
                .option("header", "true")
                .schema(SOURCE_SCHEMAS[source_name])
                .csv(file_info.path)
            )

            # The write is the only full action on the ingestion path.
            (
                df.write
                .format("delta")
                .mode("overwrite")
                .option("overwriteSchema", "true")
                .saveAsTable(full_table_name)
            )

            print(f"✓ Created {full_table_name}")

        except Exception as exc:
            failures.append(
                f"{file_info.path}: {exc}"
            )

if failures:
    raise RuntimeError(
        "Bronze ingestion failed:\n - "
        + "\n - ".join(failures)
    )

print("Bronze ingestion completed.")


## Bronze diagnostics

Row count and every column's null count are computed in one aggregation
per table instead of one full scan per column.


In [ ]:
DIAGNOSTIC_TABLE = (
    f"{CATALOG}.{BRONZE_SCHEMA}.diagnostics_bronze"
)

bronze_tables = [
    row["tableName"]
    for row in (
        spark.sql(
            f"SHOW TABLES IN {CATALOG}.{BRONZE_SCHEMA}"
        )
        .filter("tableName LIKE 'bronze_%'")
        .collect()
    )
]

diagnostics = []

for table_name in bronze_tables:
    full_name = (
        f"{CATALOG}.{BRONZE_SCHEMA}.{table_name}"
    )
    df = spark.table(full_name)
    columns = df.columns
    column_count = len(columns)

    expressions = [
        F.count(F.lit(1))
        .cast("long")
        .alias("_row_count")
    ]

    for i, column_name in enumerate(columns):
        expressions.append(
            F.sum(
                F.when(
                    F.col(column_name).isNull(),
                    F.lit(1),
                ).otherwise(F.lit(0))
            )
            .cast("long")
            .alias(f"_null_{i}")
        )

    metrics = df.agg(*expressions).first()

    row_count = int(metrics["_row_count"] or 0)
    null_counts = {
        column_name: int(metrics[f"_null_{i}"] or 0)
        for i, column_name in enumerate(columns)
    }

    total_nulls = sum(null_counts.values())
    columns_with_nulls = sum(
        value > 0
        for value in null_counts.values()
    )
    total_cells = row_count * column_count

    completeness = (
        100.0
        if total_cells == 0
        else round(
            100.0
            * (total_cells - total_nulls)
            / total_cells,
            4,
        )
    )

    diagnostics.append({
        "table_name": table_name,
        "row_count": row_count,
        "column_count": column_count,
        "total_null_values": total_nulls,
        "columns_with_nulls": columns_with_nulls,
        "completeness_percentage": completeness,
        "null_details": json.dumps(
            {
                c: n
                for c, n in null_counts.items()
                if n > 0
            },
            sort_keys=True,
        ),
    })

diagnostic_schema = StructType([
    StructField("table_name", StringType(), False),
    StructField("row_count", LongType(), False),
    StructField("column_count", LongType(), False),
    StructField("total_null_values", LongType(), False),
    StructField("columns_with_nulls", LongType(), False),
    StructField(
        "completeness_percentage",
        DoubleType(),
        False,
    ),
    StructField("null_details", StringType(), False),
])

diagnostics_df = (
    spark.createDataFrame(
        diagnostics,
        schema=diagnostic_schema,
    )
    .orderBy("table_name")
)

(
    diagnostics_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(DIAGNOSTIC_TABLE)
)

print(f"Saved Bronze diagnostics: {DIAGNOSTIC_TABLE}")
